# Module 24 — Enterprise MCP Server

Build a governed capability server. This is an educational implementation that focuses on contracts, authorization, tenant isolation, idempotency, audit and failure handling.

In [ ]:
from dataclasses import dataclass
import hashlib, json
@dataclass(frozen=True)
class Principal:
    subject:str; tenant:str; roles:frozenset[str]
@dataclass(frozen=True)
class Capability:
    name:str; risk:str='low'; roles:frozenset[str]=frozenset()


## 1. Capability catalog
Create read, resource and mutation capabilities. Keep high-risk capabilities separate.

In [ ]:
catalog=[Capability('read_customer',roles=frozenset({'reader','admin'})),Capability('update_customer','high',frozenset({'admin'}))]
catalog


## 2. Authorization
Reject cross-tenant requests before the handler is invoked. Then enforce role permissions.

In [ ]:
p=Principal('alice','tenant-a',frozenset({'reader'}))
assert p.tenant=='tenant-a' and 'reader' in p.roles
print('principal validated')


## 3. Canonical action identity
Build a stable hash from tenant + capability + canonical arguments. This is useful for audit and exact-action approval binding.

In [ ]:
payload={'tenant':'tenant-a','capability':'update_customer','arguments':{'id':'c1','status':'active'}}
action_hash=hashlib.sha256(json.dumps(payload,sort_keys=True,separators=(',',':')).encode()).hexdigest()
print(action_hash)


## 4. Idempotent mutation
Execute a mutation twice with the same idempotency key. Expected: one external side effect and a reusable result.

In [ ]:
effects={}
key='mutation-001'
effects[key]={'status':'updated'}
print(effects[key])


## 5. Approval gate
Design an approval containing exact action hash, approver, expiry and tenant. Change one argument and demonstrate that the approval no longer matches.

## 6. Resource authorization
Create tenant-scoped resource URIs. Test that tenant-b cannot read tenant-a's resource.

## 7. Failure injection
Inject: malformed arguments, unknown capability, role denial, tenant mismatch, expired approval, duplicate mutation, downstream timeout and secret leakage. Define fail-closed behavior for each.

## 8. Security red team
Try to override policy through tool output, use a cross-tenant identifier, replay a mutation, and smuggle an external URL. Document which control must stop each attempt.

## 9. Module integration
Connect the server to Module 23 client, Module 18 security controls, Module 21 coordination and Module 22 distributed debugging. Every execution should have correlation and causation identifiers.

## 15 extension exercises
1. Add schema validation.
2. Add capability versions.
3. Add authentication.
4. Add approval expiry.
5. Add rate limits.
6. Add timeout budgets.
7. Add retry classification.
8. Add resource URI policy.
9. Add audit JSONL.
10. Add safe replay fixtures.
11. Add downstream circuit breaker.
12. Add capability emergency revocation.
13. Add multi-region failover simulation.
14. Add security regression tests.
15. Build the enterprise gateway gold challenge.


# Gold challenge
Build an enterprise MCP gateway where capabilities are discovered safely, authorized by tenant/role, high-impact actions require exact approval, mutations are idempotent, traces are auditable and failures are fail-closed.